# Static FFNN training: SA-AKI static representation

The second of four training-setup notebooks.

Input is `data/static_representation.parquet` from preprocessing, the same file
used by the XGBoost notebook: one row per stay, aggregate features, label
attached.

Two imbalance-handling conditions are run as complete, independent pipelines,
each with its own Optuna search, refit and test evaluation. The train/test
split is shared between them.

- No imbalance handling: the model is trained without class weights.
- Class weighting: Keras `class_weight` computed from training class counts at
  each fit.

## Setup

Imports and file paths. The fit/transform and evaluation functions are imported
from `src/`, the same versions used by the XGBoost notebook.

In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import optuna
import keras
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    recall_score, precision_score, f1_score,
)

sys.path.insert(0, str(Path.cwd().parent))
from src.preprocessing_fit_transform import (
    winsorize_fit, winsorize_transform,
    impute_fit, impute_transform,
    scale_fit, scale_transform,
    encode,
)
from src.evaluation_utils import (
    calibration_report, plot_calibration,
    bootstrap_ci, bootstrap_metrics_report,
    subgroup_report,
    decision_curve, plot_decision_curve,
)

DATA_DIR = Path("../data")
DATA_PATH = DATA_DIR / "static_representation.parquet"

RANDOM_STATE = 42
N_TRIALS = 50
N_FOLDS = 5
EPOCH_CEILING = 200
EARLY_STOPPING_PATIENCE = 18
REFIT_VAL_FRACTION = 0.15

CONDITIONS = ["no imbalance handling", "class weighting"]

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("keras", keras.__version__, "| optuna", optuna.__version__)

## 1. Load the data

Prints shape and class balance, and checks the column names against what
preprocessing produces.

In [ ]:
df = pd.read_parquet(DATA_PATH)

print("shape:", df.shape)
print("stays:", df["stay_id"].nunique(), "| patients:", df["subject_id"].nunique())
print("\nclass balance:")
print(df["label"].value_counts().rename("count"))
print(f"positive rate: {df['label'].mean() * 100:.2f}%")
df.head()

### Confirm the columns match what is expected

Aggregate feature columns follow the `{feature}_{stat}` naming from
preprocessing, with stats min, max, mean and last. Identifier, categorical and
label columns are checked to be present.

In [ ]:
STATS = ["min", "max", "mean", "last"]
AGG_COLS = [c for c in df.columns if c.rsplit("_", 1)[-1] in STATS]
REQUIRED = ["stay_id", "subject_id", "admission_age", "gender", "admission_type", "label"]

missing = [c for c in REQUIRED if c not in df.columns]
unaccounted = sorted(set(df.columns) - set(AGG_COLS) - set(REQUIRED))

print(f"aggregate feature columns: {len(AGG_COLS)}  (expected 22 features x 4 stats = 88)")
print("missing required columns:", missing if missing else "none")
print("unaccounted columns:", unaccounted if unaccounted else "none")

assert not missing, f"missing required columns: {missing}"
assert not unaccounted, f"unexpected columns: {unaccounted}"

## 2. Separate features from label and identifiers

`label` is the target. `stay_id` and `subject_id` are identifiers and are
excluded from the model. `gender` and `admission_type` are handled by
`encode()`. The remaining numeric columns, the aggregates plus `admission_age`,
are passed to Winsorize, imputation and scaling.

In [ ]:
TARGET = "label"
ID_COLS = ["stay_id", "subject_id"]
CATEGORICAL = ["gender", "admission_type"]
FEATURE_COLS = AGG_COLS + ["admission_age"]   # winsorize/impute/scale apply here

DROP_FROM_X = ID_COLS + [TARGET]

print(f"numeric feature columns (winsorize/impute/scale): {len(FEATURE_COLS)}")
print(f"categorical (encoded): {CATEGORICAL}")
print(f"identifiers (excluded from X): {ID_COLS}")

## 3. Train/test split

80/20, stratified by `label`, `random_state=42`. The static representation is
one row per patient, so no patient-level grouping is applied.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df[TARGET], random_state=RANDOM_STATE
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"train: {len(train_df):,} rows, positive rate {train_df[TARGET].mean() * 100:.2f}%")
print(f"test:  {len(test_df):,} rows, positive rate {test_df[TARGET].mean() * 100:.2f}%")
print(f"\npatient overlap between train and test: "
      f"{len(set(train_df['subject_id']) & set(test_df['subject_id']))}")

## 4. Fold processing and class weights

Winsorize, imputation and scaling are fitted on the training portion passed to
this function and applied unchanged to the other portion. `encode()` is then
applied to both. The same function serves the inner CV folds and the final
refit.

Class weights follow `n_samples / (n_classes * n_samples_in_class)` and are
recomputed wherever they are used rather than fixed in advance.

In [ ]:
def process(fit_df, apply_df, feature_cols=FEATURE_COLS):
    """Fit Winsorize/impute/scale on fit_df, apply to both frames, then encode.

    Returns (fit_df_processed, apply_df_processed, fitted_params).
    """
    bounds = winsorize_fit(fit_df, feature_cols)
    fit_df = winsorize_transform(fit_df, bounds)
    apply_df = winsorize_transform(apply_df, bounds)

    medians = impute_fit(fit_df, feature_cols)
    fit_df = impute_transform(fit_df, medians)
    apply_df = impute_transform(apply_df, medians)

    scaler = scale_fit(fit_df, feature_cols)
    fit_df = scale_transform(fit_df, scaler, feature_cols)
    apply_df = scale_transform(apply_df, scaler, feature_cols)

    return encode(fit_df), encode(apply_df), {
        "bounds": bounds, "medians": medians, "scaler": scaler,
    }


def to_xy(processed_df):
    """Split a processed frame into a float32 model matrix and the target."""
    X = processed_df.drop(columns=DROP_FROM_X).to_numpy(dtype="float32")
    y = processed_df[TARGET].to_numpy(dtype="float32")
    return X, y


def class_weights(y):
    """Return n_samples / (n_classes * n_samples_in_class) for each class."""
    counts = np.bincount(y.astype(int), minlength=2)
    n_samples, n_classes = len(y), 2
    return {c: float(n_samples / (n_classes * counts[c])) for c in (0, 1)}

## 5. Model architecture

A feedforward network: 1 to 3 hidden Dense layers with ReLU activation and L2
weight regularization, each followed by Dropout, then a single sigmoid output
unit. Optimizer Adam, loss binary crossentropy, with AUPRC tracked as a metric
so early stopping can monitor it.

In [ ]:
def build_model(n_features, n_layers, units, dropout, l2_value, learning_rate):
    """Build and compile the feedforward network. Returns the compiled model."""
    model = keras.Sequential(name="static_ffnn")
    model.add(keras.layers.Input(shape=(n_features,)))

    for _ in range(n_layers):
        model.add(keras.layers.Dense(
            units, activation="relu",
            kernel_regularizer=keras.regularizers.l2(l2_value),
        ))
        if dropout > 0:
            model.add(keras.layers.Dropout(dropout))

    model.add(keras.layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(curve="PR", name="auprc")],
    )
    return model


def early_stopping():
    """Return an EarlyStopping callback monitoring validation AUPRC."""
    return keras.callbacks.EarlyStopping(
        monitor="val_auprc", mode="max",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    )

## 6. Optuna objective and search space

Hyperparameters are selected on inner folds and evaluated on a test set held
out of selection (Cawley & Talbot, 2010; Varma & Simon, 2006).

Each trial builds five stratified folds from the training split. Winsorize,
imputation and scaling are fitted on each fold's own training portion. For
class weighting, the weight dictionary is recomputed from that fold's training
labels. Training runs to a ceiling of 200 epochs with early stopping on the
validation fold, monitoring AUPRC.

Mean fold AUPRC is reported after each fold, allowing a trial to be pruned
before all five folds run. The epoch reaching the best validation AUPRC is
stored on the trial for use at refit.

| Parameter | Status | Range / value |
|---|---|---|
| Number of hidden layers | Tuned | 1 to 3, integer |
| Units per layer | Tuned | 8 to 128, integer, log scale |
| Dropout rate | Tuned | 0.0 to 0.5 |
| L2 weight regularization | Tuned | 1e-6 to 1e-2, log scale |
| Learning rate | Tuned | 1e-4 to 1e-2, log scale |
| Batch size | Tuned | 16, 32, 64 or 128, categorical |
| Epochs | Fixed | Ceiling 200, early stopping |
| `class_weight` | Fixed | Computed from class counts, class weighting only |
| Activation | Fixed | ReLU |
| Optimizer | Fixed | Adam |

In [ ]:
def suggest_params(trial):
    return {
        "n_layers": trial.suggest_int("n_layers", 1, 3),
        "units": trial.suggest_int("units", 8, 128, log=True),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "l2_value": trial.suggest_float("l2_value", 1e-6, 1e-2, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64, 128]),
    }


def make_objective(condition):
    """Return the Optuna objective for the given imbalance-handling condition."""

    def objective(trial):
        params = suggest_params(trial)
        batch_size = params.pop("batch_size")
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

        fold_scores, fold_best_epochs = [], []
        for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df[TARGET])):
            fold_train = train_df.iloc[tr_idx].reset_index(drop=True)
            fold_valid = train_df.iloc[va_idx].reset_index(drop=True)

            fold_train, fold_valid, _ = process(fold_train, fold_valid)
            X_tr, y_tr = to_xy(fold_train)
            X_va, y_va = to_xy(fold_valid)

            weights = class_weights(y_tr) if condition == "class weighting" else None

            keras.utils.set_random_seed(RANDOM_STATE)
            model = build_model(X_tr.shape[1], **params)
            history = model.fit(
                X_tr, y_tr,
                validation_data=(X_va, y_va),
                epochs=EPOCH_CEILING,
                batch_size=batch_size,
                class_weight=weights,
                callbacks=[early_stopping()],
                verbose=0,
            )

            proba = model.predict(X_va, verbose=0).ravel()
            fold_scores.append(average_precision_score(y_va, proba))
            fold_best_epochs.append(int(np.argmax(history.history["val_auprc"])) + 1)

            keras.backend.clear_session()

            trial.report(float(np.mean(fold_scores)), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        trial.set_user_attr("mean_best_epoch", int(np.mean(fold_best_epochs)))
        return float(np.mean(fold_scores))

    return objective

## 7. Run the Optuna study, once per condition

50 trials per condition, maximising mean fold AUPRC, with median pruning
(Akiba et al., 2019). The two studies are independent.

In [ ]:
def run_study(condition):
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        study_name=f"static_ffnn_{condition.lower().replace(' ', '_')}",
    )
    study.optimize(make_objective(condition), n_trials=N_TRIALS, show_progress_bar=True)

    pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    print(f"\n{condition}: {len(study.trials)} trials, {pruned} pruned")
    print(f"best mean CV AUPRC: {study.best_value:.4f}")
    print(f"best epoch (mean over folds): {study.best_trial.user_attrs['mean_best_epoch']}")
    print("best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    return study

### No imbalance handling

In [ ]:
study_no_weighting = run_study("no imbalance handling")

### Class weighting

In [ ]:
study_class_weighting = run_study("class weighting")

## 8. Refit on the full training split

For each condition, Winsorize, imputation and scaling are fitted once on the
entire 80% training split. One model per condition is then trained on that
processed data using the condition's winning hyperparameters.

Early stopping needs a validation set, so a stratified 15% of the training
split is held back for it. The class weight dictionary is computed from the
full training split. The test set takes no part in this step.

In [ ]:
def refit(study, condition):
    params = dict(study.best_params)
    batch_size = params.pop("batch_size")

    train_proc, test_proc, fitted = process(train_df.copy(), test_df.copy())
    X_train, y_train = to_xy(train_proc)

    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train, y_train, test_size=REFIT_VAL_FRACTION,
        stratify=y_train, random_state=RANDOM_STATE,
    )

    weights = class_weights(y_train) if condition == "class weighting" else None

    keras.utils.set_random_seed(RANDOM_STATE)
    model = build_model(X_fit.shape[1], **params)
    history = model.fit(
        X_fit, y_fit,
        validation_data=(X_val, y_val),
        epochs=EPOCH_CEILING,
        batch_size=batch_size,
        class_weight=weights,
        callbacks=[early_stopping()],
        verbose=0,
    )

    best_epoch = int(np.argmax(history.history["val_auprc"])) + 1
    print(f"{condition}: refit on {len(X_fit):,} rows "
          f"({len(X_val):,} held back for early stopping), best epoch {best_epoch}"
          + (f", class weights {weights}" if weights else ""))
    return model, test_proc, fitted, best_epoch


model_no_weighting, test_proc_no_weighting, fitted_no_weighting, epoch_no_weighting = refit(
    study_no_weighting, "no imbalance handling")
model_class_weighting, test_proc_class_weighting, fitted_class_weighting, epoch_class_weighting = refit(
    study_class_weighting, "class weighting")

## 9. Evaluate on the held-out test set

The Winsorize bounds, medians and scaler fitted in step 8 are applied to the
test set unchanged. Nothing is refitted on test. AUPRC is the primary metric,
with AUROC, recall, specificity, precision and F1 reported at a 0.5 decision
threshold.

In [ ]:
def evaluate(model, test_proc, condition):
    X_te, y_te = to_xy(test_proc)
    proba = model.predict(X_te, verbose=0).ravel()
    pred = (proba >= 0.5).astype(int)

    return proba, {
        "condition": condition,
        "AUPRC": average_precision_score(y_te, proba),
        "AUROC": roc_auc_score(y_te, proba),
        "recall_AKI": recall_score(y_te, pred),                    # positive class
        "specificity_non_AKI": recall_score(y_te, pred, pos_label=0),  # negative class
        "precision": precision_score(y_te, pred),
        "F1": f1_score(y_te, pred),
    }


proba_no_weighting, metrics_no_weighting = evaluate(
    model_no_weighting, test_proc_no_weighting, "no imbalance handling")
proba_class_weighting, metrics_class_weighting = evaluate(
    model_class_weighting, test_proc_class_weighting, "class weighting")

_, y_test = to_xy(test_proc_no_weighting)

print(f"test rows: {len(y_test):,}")
print(f"test positive rate: {y_test.mean() * 100:.2f}%  "
      f"(AUPRC of a random baseline equals the positive rate)")

## 10. Summary table

One row per condition.

In [ ]:
summary = (
    pd.DataFrame([metrics_no_weighting, metrics_class_weighting])
      .set_index("condition")
      .round(4)
)
summary

## 11. Calibration of the predicted probabilities

Calibration compares predicted probabilities against the rates observed at
those probabilities, where AUPRC and AUROC measure ranking.

Predictions come from the models refitted in step 8. Nothing is refitted,
retrained or resampled here. Both conditions are drawn on one chart.

In [ ]:
bins_no_weighting = calibration_report(y_test, proba_no_weighting, label="no imbalance handling")
bins_class_weighting = calibration_report(y_test, proba_class_weighting, label="class weighting")

fig_calibration = plot_calibration([(bins_no_weighting, "no imbalance handling"), (bins_class_weighting, "class weighting")])

## 12. Bootstrap confidence intervals

The test set is resampled with replacement and each metric from step 9 is
recomputed on every resample, giving an interval around each point estimate.

Predictions come from the models refitted in step 8. Only the test set rows are
resampled.

In [ ]:
from IPython.display import display

boot_no_weighting = bootstrap_metrics_report(y_test, proba_no_weighting, threshold=0.5,
                                             label="no imbalance handling")
boot_class_weighting = bootstrap_metrics_report(y_test, proba_class_weighting, threshold=0.5,
                                                label="class weighting")

display(boot_no_weighting)
display(boot_class_weighting)

## 13. Subgroup performance

Performance is reported separately by gender, age band and admission type, each
with the bootstrap intervals used in step 12. Group sizes are reported
alongside, and subgroup intervals are wider than the overall result.

Predictions come from the models refitted in step 8. Nothing is refitted or
retrained.

In [ ]:
# Subgroup labels come from the untransformed test split: admission_age in
# test_proc_* is scaled, so age banding needs the raw values. Row order is
# unchanged, so these line up with y_test and the predicted probabilities.
gender_values = test_df["gender"]

age_bands = pd.cut(test_df["admission_age"], bins=[18, 40, 65, 90],
                   labels=["18-40", "40-65", "65-90"], include_lowest=True)

# the encoded emergency/elective split; several raw admission_type categories
# are too small to bootstrap
admission_groups = test_proc_no_weighting["admission_type_elective"].map(
    {0: "Emergency", 1: "Elective"}
)

print("group sizes")
for name, values in [("gender", gender_values), ("age band", age_bands),
                     ("admission type", admission_groups)]:
    counts = values.value_counts(dropna=False).sort_index().to_dict()
    print(f"  {name}: {counts}")
print()

In [ ]:
SUBGROUPS = {
    "Gender": gender_values,
    "Age band": age_bands,
    "Admission type": admission_groups,
}
CONDITION_PROBA = {
    "no imbalance handling": proba_no_weighting,
    "class weighting": proba_class_weighting,
}

subgroup_tables = {}
for attribute, values in SUBGROUPS.items():
    for condition, proba in CONDITION_PROBA.items():
        key = f"{condition} -- {attribute}"
        subgroup_tables[key] = subgroup_report(y_test, proba, values, label=key)

### Subgroup results

One table per condition and attribute, each row a metric within one subgroup,
with the point estimate, the confidence interval and the subgroup size.

In [ ]:
for key, table in subgroup_tables.items():
    print(key)
    display(table)

## 14. Decision curve analysis

Net benefit is computed across the full range of thresholds and compared
against two reference strategies, flagging every patient and flagging none
(Vickers & Elkin, 2006). A model is ahead over the range where its curve sits
above both reference lines.

Predictions come from the models refitted in step 8. Nothing is refitted or
retrained.

In [ ]:
curve_no_weighting = decision_curve(y_test, proba_no_weighting)
curve_class_weighting = decision_curve(y_test, proba_class_weighting)

fig_dca = plot_decision_curve([(curve_no_weighting, "no imbalance handling"),
                               (curve_class_weighting, "class weighting")])

print(curve_no_weighting[(curve_no_weighting["threshold"] >= 0.4) & (curve_no_weighting["threshold"] <= 0.7)].round(4).to_string(index=False))
print()
print(curve_class_weighting[(curve_class_weighting["threshold"] >= 0.4) & (curve_class_weighting["threshold"] <= 0.7)].round(4).to_string(index=False))

## Output

Per condition: winning hyperparameters, test metrics, and the fitted Winsorize
bounds, medians and scaler parameters. Written to `data/`, which is gitignored.
The scaler is stored as its centre and scale arrays rather than the object
itself.

In [ ]:
def save_results(study, metrics, fitted, best_epoch, condition):
    payload = {
        "condition": condition,
        "best_params": study.best_params,
        "epoch_used_at_refit": best_epoch,
        "best_cv_auprc": study.best_value,
        "test_metrics": {k: v for k, v in metrics.items() if k != "condition"},
        "fitted_params": {
            "winsorize_bounds": {k: [float(lo), float(hi)] for k, (lo, hi) in fitted["bounds"].items()},
            "medians": {k: float(v) for k, v in fitted["medians"].items()},
            "scaler_center": [float(v) for v in fitted["scaler"].center_],
            "scaler_scale": [float(v) for v in fitted["scaler"].scale_],
            "scaler_feature_order": FEATURE_COLS,
        },
    }
    path = DATA_DIR / f"static_ffnn_{condition.lower().replace(' ', '_')}_results.json"
    path.write_text(json.dumps(payload, indent=2))
    print(f"wrote {path.name}  ({path.stat().st_size / 1024:.1f} KB)")


save_results(study_no_weighting, metrics_no_weighting, fitted_no_weighting,
             epoch_no_weighting, "no imbalance handling")
save_results(study_class_weighting, metrics_class_weighting, fitted_class_weighting,
             epoch_class_weighting, "class weighting")